In [1]:
!pip install -q transformers datasets accelerate evaluate streamlit pandas scikit-learn matplotlib

In [2]:
import os, re, json, html, time, random, unicodedata, string, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import GroupShuffleSplit
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer, default_data_collator, pipeline)

In [3]:
MODELS = ["distilbert-base-uncased", "bert-base-uncased"]

In [4]:
df = pd.read_csv('/content/SQuAD-v1.1.csv')

print(df.shape)
df.head()

(87599, 6)


,title,context,question,answer,answer_start,answer_end
0,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,Saint Bernadette Soubirous,515,541
1,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,a copper statue of Christ,188,213
2,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,the Main Building,279,296
3,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is the Grotto at Notre Dame?,a Marian place of prayer and reflection,381,420
4,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What sits on top of the Main Building at Notre...,a golden statue of the Virgin Mary,92,126


In [5]:
print(df.dtypes, "\n")
print("Missing values:\n", df.isna().sum(), "\n")
print("Duplicate rows:", df.duplicated().sum())
print("Unique contexts:", df["context"].nunique(), "| unique titles:", df["title"].nunique())
for c in ["context", "question", "answer"]:
    print(f"{c:9s} avg length (chars): {df[c].astype(str).str.len().mean():.1f}")

title           object
context         object
question        object
answer          object
answer_start     int64
answer_end       int64
dtype: object 

Missing values:
 title           0
context         0
question        0
answer          3
answer_start    0
answer_end      0
dtype: int64 

Duplicate rows: 92
Unique contexts: 18891 | unique titles: 442
context   avg length (chars): 754.4
question  avg length (chars): 59.6
answer    avg length (chars): 20.1


In [6]:
TAG_RE  = re.compile(r"<[^>]+>")
CTRL_RE = re.compile(r"[\u0000-\u0008\u000b\u000c\u000e-\u001f\u007f\u200b\u200c\u200d\ufeff]")
WS_RE   = re.compile(r"\s+")

def clean_text(s, strip=True):
    s = html.unescape(str(s))
    s = TAG_RE.sub(" ", s)
    s = CTRL_RE.sub("", s)
    s = unicodedata.normalize("NFKC", s)
    s = WS_RE.sub(" ", s)
    return s.strip() if strip else s

def preprocess(df):
    n0 = len(df)
    df = df.dropna(subset=["context", "question", "answer", "answer_start"]).drop_duplicates().copy()
    df["answer_start"] = df["answer_start"].astype(int)

    new_ctx, new_ans, new_start = [], [], []
    for ctx, ans, st in zip(df["context"], df["answer"], df["answer_start"]):
        ctx, ans = str(ctx), str(ans)
        c_clean = clean_text(ctx)
        a_clean = clean_text(ans)

        prefix = clean_text(ctx[:st], strip=False).lstrip()
        ns = len(prefix)

        found = -1
        for d in (0, 1, -1, 2, -2):
            if ns + d >= 0 and c_clean[ns + d: ns + d + len(a_clean)] == a_clean:
                found = ns + d; break
        new_ctx.append(c_clean); new_ans.append(a_clean); new_start.append(found)

    df["context"], df["answer"], df["answer_start"] = new_ctx, new_ans, new_start
    df["question"] = df["question"].map(clean_text)
    df["title"] = df["title"].fillna("unknown").map(clean_text)
    df = df[(df["answer_start"] >= 0) & (df["answer"].str.len() > 0) & (df["question"].str.len() > 0)]
    df["answer_end"] = df["answer_start"] + df["answer"].str.len()
    df = df.reset_index(drop=True)
    df["id"] = [f"q{i}" for i in range(len(df))]
    print(f"Rows: {n0} -> {len(df)} after cleaning/validation")
    return df

df = preprocess(df)
assert all(c[s:e] == a for c, a, s, e in zip(df.context, df.answer, df.answer_start, df.answer_end))
df.head(3)

Rows: 87599 -> 87504 after cleaning/validation


,title,context,question,answer,answer_start,answer_end,id
0,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,Saint Bernadette Soubirous,515,541,q0
1,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,a copper statue of Christ,188,213,q1
2,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,the Main Building,279,296,q2


In [7]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
tr_idx, va_idx = next(gss.split(df, groups=df["title"]))
train_df, val_df = df.iloc[tr_idx].reset_index(drop=True), df.iloc[va_idx].reset_index(drop=True)

train_df = train_df.sample(len(train_df), random_state=42).reset_index(drop=True)
val_df = val_df.sample(len(val_df), random_state=42).reset_index(drop=True)
print("train:", len(train_df), "| val:", len(val_df))

cols = ["id", "context", "question", "answer", "answer_start"]
train_raw = Dataset.from_pandas(train_df[cols], preserve_index=False)
val_raw = Dataset.from_pandas(val_df[cols],   preserve_index=False)

train: 77576 | val: 9928


In [8]:
def make_prepare_fns(tokenizer):
    def prepare_train(examples):
        questions = [q.lstrip() for q in examples["question"]]
        tok = tokenizer(questions, examples["context"], truncation="only_second",
                        max_length=384, stride=128,
                        return_overflowing_tokens=True, return_offsets_mapping=True,
                        padding="max_length")
        sample_map = tok.pop("overflow_to_sample_mapping")
        offsets_all = tok.pop("offset_mapping")
        tok["start_positions"], tok["end_positions"] = [], []

        for i, offsets in enumerate(offsets_all):
            input_ids = tok["input_ids"][i]
            cls_idx = input_ids.index(tokenizer.cls_token_id)
            seq_ids = tok.sequence_ids(i)
            s_idx = sample_map[i]
            start_char = examples["answer_start"][s_idx]
            end_char = start_char + len(examples["answer"][s_idx])

            ts = 0
            while seq_ids[ts] != 1: ts += 1
            te = len(input_ids) - 1
            while seq_ids[te] != 1: te -= 1

            if not (offsets[ts][0] <= start_char and offsets[te][1] >= end_char):
                tok["start_positions"].append(cls_idx); tok["end_positions"].append(cls_idx)
            else:
                while ts < len(offsets) and offsets[ts][0] <= start_char: ts += 1
                tok["start_positions"].append(ts - 1)
                while offsets[te][1] >= end_char: te -= 1
                tok["end_positions"].append(te + 1)
        return tok

    def prepare_val(examples):
        questions = [q.lstrip() for q in examples["question"]]
        tok = tokenizer(questions, examples["context"], truncation="only_second",
                        max_length=384, stride=128,
                        return_overflowing_tokens=True, return_offsets_mapping=True,
                        padding="max_length")
        sample_map = tok.pop("overflow_to_sample_mapping")
        tok["example_id"] = []
        for i in range(len(tok["input_ids"])):
            seq_ids = tok.sequence_ids(i)
            tok["example_id"].append(examples["id"][sample_map[i]])
            tok["offset_mapping"][i] = [o if seq_ids[k] == 1 else None
                                        for k, o in enumerate(tok["offset_mapping"][i])]
        return tok
    return prepare_train, prepare_val

_tk = AutoTokenizer.from_pretrained(MODELS[0])
_pt, _ = make_prepare_fns(_tk)
_f = _pt(train_raw[:2])
s, e = _f["start_positions"][0], _f["end_positions"][0]
print("Gold answer :", train_raw[0]["answer"])
print("Decoded span:", _tk.decode(_f["input_ids"][0][s:e + 1]))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Gold answer : lavish costume balls
Decoded span: lavish costume balls


In [9]:
def postprocess(examples, features, raw_predictions, n_best=20, max_answer_len=30):
    start_logits_all, end_logits_all = raw_predictions
    id_to_idx = {k: i for i, k in enumerate(examples["id"])}
    feats_per_ex = collections.defaultdict(list)
    for i, f in enumerate(features):
        feats_per_ex[id_to_idx[f["example_id"]]].append(i)

    preds = {}
    for ex_i, ex in enumerate(examples):
        context, best = ex["context"], None
        for fi in feats_per_ex[ex_i]:
            sl, el = start_logits_all[fi], end_logits_all[fi]
            offsets = features[fi]["offset_mapping"]
            for s in np.argsort(sl)[-1:-n_best - 1:-1]:
                for e in np.argsort(el)[-1:-n_best - 1:-1]:
                    if s >= len(offsets) or e >= len(offsets): continue
                    if offsets[s] is None or offsets[e] is None: continue
                    if e < s or e - s + 1 > max_answer_len: continue
                    score = sl[s] + el[e]
                    if best is None or score > best["score"]:
                        best = {"score": float(score), "text": context[offsets[s][0]:offsets[e][1]]}
        preds[ex["id"]] = best["text"] if best else ""
    return preds

def normalize_answer(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def f1_score(pred, gold):
    p, g = normalize_answer(pred).split(), normalize_answer(gold).split()
    common = collections.Counter(p) & collections.Counter(g)
    n_same = sum(common.values())
    if len(p) == 0 or len(g) == 0:
        return float(p == g)
    if n_same == 0: return 0.0
    prec, rec = n_same / len(p), n_same / len(g)
    return 2 * prec * rec / (prec + rec)

def evaluate_predictions(preds, examples):
    ems, f1s = [], []
    for ex in examples:
        p = preds[ex["id"]]
        ems.append(exact_match(p, ex["answer"])); f1s.append(f1_score(p, ex["answer"]))
    return {"exact_match": 100 * np.mean(ems), "f1": 100 * np.mean(f1s)}

print(exact_match("The Eiffel Tower", "eiffel tower"), round(f1_score("the big Eiffel Tower", "Eiffel Tower"), 3))

1.0 0.8


In [10]:
def train_and_eval(model_name):
    print(f"\n{'=' * 70}\n  {model_name}\n{'=' * 70}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForQuestionAnswering.from_pretrained(model_name)
    prep_train, prep_val = make_prepare_fns(tokenizer)

    train_feats = train_raw.map(prep_train, batched=True, remove_columns=train_raw.column_names)
    val_feats   = val_raw.map(prep_val,   batched=True, remove_columns=val_raw.column_names)

    args = TrainingArguments(
        output_dir=os.path.join('/content/output', model_name.replace("/", "_")),
        learning_rate=3e-5, num_train_epochs=2,
        per_device_train_batch_size=16, per_device_eval_batch_size=16 * 2,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        save_strategy="no", report_to="none", seed=42, logging_steps=100,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_feats,
                      data_collator=default_data_collator, processing_class=tokenizer)

    t0 = time.time(); trainer.train(); train_time = time.time() - t0

    t0 = time.time()
    raw = trainer.predict(val_feats.remove_columns(["example_id", "offset_mapping"]))
    preds = postprocess(val_raw, val_feats, raw.predictions)
    infer_time = time.time() - t0

    metrics = evaluate_predictions(preds, val_raw)
    metrics.update({"model": model_name, "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
                    "train_min": train_time / 60, "infer_sec": infer_time})
    print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in metrics.items()})
    return metrics, trainer, tokenizer, preds

In [11]:
results, artifacts = [], {}
for name in MODELS:
    m, trainer, tok, preds = train_and_eval(name)
    results.append(m)
    artifacts[name] = (trainer, tok, preds)
    torch.cuda.empty_cache()

res_df = (pd.DataFrame(results)[["model", "exact_match", "f1", "params_M", "train_min", "infer_sec"]]
          .sort_values("f1", ascending=False).reset_index(drop=True).round(2))
res_df


  distilbert-base-uncased


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/77576 [00:00<?, ? examples/s]

... (12 outputs left out)

Map:   0%|          | 0/9928 [00:00<?, ? examples/s]

Step,Training Loss
100,4.115011
200,2.528765
300,1.880985
400,1.817244
500,1.634106
600,1.590713
700,1.429609
800,1.462955
900,1.424471
1000,1.436533


{'exact_match': np.float64(68.87), 'f1': np.float64(81.83), 'model': 'bert-base-uncased', 'params_M': 108.89, 'train_min': 48.75, 'infer_sec': 94.93}


,model,exact_match,f1,params_M,train_min,infer_sec
0,bert-base-uncased,68.87,81.83,108.89,48.75,94.93
1,distilbert-base-uncased,65.76,78.90,66.36,24.75,64.35


In [12]:
cols = ["model", "exact_match", "f1", "train_min"]
table_df = res_df[cols].copy()

table_df[["exact_match", "f1"]] = table_df[["exact_match", "f1"]].round(2)
table_df["train_min"] = table_df["train_min"].round(2)

display(table_df)

print(table_df.to_string(index=False))

res_df.to_csv("model_comparison.csv", index=False)

,model,exact_match,f1,train_min
0,bert-base-uncased,68.87,81.83,48.75
1,distilbert-base-uncased,65.76,78.90,24.75


                  model  exact_match    f1  train_min
      bert-base-uncased        68.87 81.83      48.75
distilbert-base-uncased        65.76 78.90      24.75


In [13]:
best_name = res_df.loc[0, "model"]
print("Best model:", best_name)
best_preds = artifacts[best_name][2]

rows = []
for ex in val_raw:
    p = best_preds[ex["id"]]
    rows.append({"question": ex["question"], "gold": ex["answer"], "pred": p, "f1": f1_score(p, ex["answer"])})
err = pd.DataFrame(rows)
print("Exactly right:", (err.f1 == 1).mean().round(3), "| partial:", ((err.f1 > 0) & (err.f1 < 1)).mean().round(3),
      "| wrong:", (err.f1 == 0).mean().round(3))
pd.set_option("display.max_colwidth", 80)
err[err.f1 < 0.5].head(10)

Best model: bert-base-uncased
Exactly right: 0.689 | partial: 0.222 | wrong: 0.09


,question,gold,pred,f1
4,What kind of dried wood retains a small quantity of water but is considered ...,oven-dried,oven,0.000000
6,Why do cells easily stain with dyes?,electrostatic attraction,the electrostatic attraction between negatively charged cellular molecules a...,0.285714
7,The militia that fought in New Haven was lead by who?,Naphtali Daggett,Yale students,0.000000
19,What letter of the alphabet do most of the last names the men of Greecs end ...,"Greek male surnames end in -s, which is the common ending",-ou,0.000000
20,What marked the start of a new and sophisticated polity that would last into...,Mudaito Dynasty,"Kedafu, head of the Mudaito clan, seized power and established the Mudaito D...",0.307692
31,What is one parent company of the Computing-Tabulating-Recording Company?,Computing Scale Company,Tabulating Machine Company,0.333333
42,What did the United Nations take charge of with regards to child labour?,International Program on the Elimination of Child Labour (IPEC),setting the international law,0.181818
46,What genre of music is played at urs?,folk,Soul inspiring music,0.000000
47,What is the differenctiation between Catalan and Valencian?,different varieties,pronunciation,0.000000
52,Where did the British turn a blind eye to child labour?,Hong Kong,"southeast Asian colonies, such as Hong Kong",0.444444


In [14]:
SAVE_DIR = "/content"
MAX_ANS_LEN = 30

best_name = res_df.loc[0, "model"]
print("Best model:", best_name)
trainer, tok, preds = artifacts[best_name]
trainer.save_model(SAVE_DIR)
tok.save_pretrained(SAVE_DIR)
with open(os.path.join(SAVE_DIR, "info.json"), "w") as f:
    json.dump({"model": best_name}, f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = AutoModelForQuestionAnswering.from_pretrained(SAVE_DIR).to(device).eval()

def run_qa(question, context, max_answer_len=30):
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    start_probs = torch.softmax(outputs.start_logits[0], dim=-1)
    end_probs = torch.softmax(outputs.end_logits[0], dim=-1)

    seq_ids = inputs.sequence_ids(0)
    context_mask = torch.tensor([1.0 if s == 1 else 0.0 for s in seq_ids], device=device)
    start_probs = start_probs * context_mask
    end_probs = end_probs * context_mask

    score_matrix = torch.outer(start_probs, end_probs)
    n = len(start_probs)
    idx = torch.arange(n, device=device)
    valid_span = (idx.unsqueeze(1) <= idx.unsqueeze(0)) & ((idx.unsqueeze(0) - idx.unsqueeze(1)) < max_answer_len)
    score_matrix = score_matrix * valid_span

    best_idx = torch.argmax(score_matrix).item()
    start_idx, end_idx = divmod(best_idx, n)
    confidence = score_matrix[start_idx, end_idx].item()

    answer = tokenizer.decode(inputs["input_ids"][0][start_idx : end_idx + 1], skip_special_tokens=True).strip()
    return {"answer": answer, "score": confidence}

passage = ("The Amazon rainforest covers much of the Amazon basin of South America. This basin encompasses "
           "7,000,000 square kilometres, of which 5,500,000 square kilometres are covered by the rainforest. "
           "The region includes territory belonging to nine nations, with the majority contained within Brazil.")

for q in ["How large is the Amazon basin?", "Which country contains most of the rainforest?", "How many nations share it?"]:
    out = run_qa(q, passage, max_answer_len=MAX_ANS_LEN)
    print(f"Q: {q}\nA: {out['answer']}  (confidence {out['score']:.2f})\n")

Best model: bert-base-uncased


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Q: How large is the Amazon basin?
A: 7, 000, 000 square kilometres  (confidence 0.86)

Q: Which country contains most of the rainforest?
A: brazil  (confidence 0.97)

Q: How many nations share it?
A: nine  (confidence 0.86)

